# Simplified MRI Processing Pipeline (FreeSurfer-Only)
## No iBEATv2 Required - Easier to Run!

This notebook demonstrates the **simplified** structural MRI processing pipeline using only FreeSurfer.

**Input File:** `/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.nii.gz`

### Key Differences from Full Pipeline:

| Feature | **Full Pipeline** | **Simplified Pipeline** |
|---------|-------------------|-------------------------|
| Prerequisites | iBEATv2 required | ✅ None - FreeSurfer only |
| Processing Time | 20-40 hours | 6-20 hours |
| Tissue Segmentation | iBEATv2 + iFS merged | FreeSurfer native |
| Complexity | High (3 tools) | Low (1-2 tools) |
| Infant Support | Optimal (under 25mo) | Good (optional iFS) |
| Adult Support | Excellent | Excellent |

### Simplified Pipeline Overview:
1. **Load and Visualize Raw T1w Image**
2. **FreeSurfer Complete Processing** (all steps in one command!)
3. **Optional: Infant FreeSurfer Enhancement** (for ages 0-24 months)
4. **Visualize Results**
5. **Extract Measurements**

### When to Use Which Pipeline?

**Use Full Pipeline (with iBEATv2) if:**
- You need absolute best accuracy for infant cortical segmentation
- You have iBEATv2 already set up
- You're doing longitudinal infant studies (0-24 months)
- You have MATLAB available

**Use Simplified Pipeline (FreeSurfer-only) if:**
- ✅ You want to get started quickly
- ✅ You don't have iBEATv2 or MATLAB
- ✅ Your subjects are 25+ months old
- ✅ You want easier maintenance and troubleshooting
- ✅ Standard FreeSurfer accuracy is sufficient for your research

In [ ]:
# Import required libraries
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib import colors
from mpl_toolkits.mplot3d import Axes3D
import subprocess
from pathlib import Path
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

In [ ]:
# Define paths and parameters
INPUT_T1W = "/data02/share/bin-wu/data/human/brain/harvard_mri/raw/new_england/ds006169-1.0.3/sub-01/ses-03/anat/sub-01_ses-03_T1w.nii.gz"
SUBJECT_ID = "sub-01_ses-03_simplified"
AGE_MONTHS = 18  # Set to None if adult or not using infant FreeSurfer

# Set up FreeSurfer environment
SUBJECTS_DIR = "/data02/share/bin-wu/data/human/brain/harvard_mri/processed/sandbox/freesurfer_simplified"
os.makedirs(SUBJECTS_DIR, exist_ok=True)

# Check if input file exists
if os.path.exists(INPUT_T1W):
    print(f"✓ Input file found: {INPUT_T1W}")
    file_size = os.path.getsize(INPUT_T1W) / (1024**2)  # MB
    print(f"  File size: {file_size:.2f} MB")
else:
    print(f"✗ Input file NOT found: {INPUT_T1W}")
    print("  Please update the INPUT_T1W path to your actual file location.")

## Step 1: Load and Visualize Raw T1w Image

In [ ]:
def load_nifti(filepath):
    """Load NIfTI file and return image data and header."""
    img = nib.load(filepath)
    data = img.get_fdata()
    return data, img.affine, img.header

def plot_3d_slices(data, title="MRI Slices", figsize=(15, 5), cmap='gray', percentiles=(1, 99)):
    """Plot axial, sagittal, and coronal slices of 3D MRI data."""
    mid_x = data.shape[0] // 2
    mid_y = data.shape[1] // 2
    mid_z = data.shape[2] // 2
    
    vmin, vmax = np.percentile(data[data > 0], percentiles)
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    axes[0].imshow(np.rot90(data[mid_x, :, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0].set_title(f'Sagittal (X={mid_x})')
    axes[0].axis('off')
    
    axes[1].imshow(np.rot90(data[:, mid_y, :]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[1].set_title(f'Coronal (Y={mid_y})')
    axes[1].axis('off')
    
    axes[2].imshow(np.rot90(data[:, :, mid_z]), cmap=cmap, vmin=vmin, vmax=vmax)
    axes[2].set_title(f'Axial (Z={mid_z})')
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig

# Load original T1w image
if os.path.exists(INPUT_T1W):
    print("Loading original T1w image...")
    t1w_data, t1w_affine, t1w_header = load_nifti(INPUT_T1W)
    
    print(f"\nImage dimensions: {t1w_data.shape}")
    print(f"Voxel size: {t1w_header.get_zooms()[:3]} mm")
    print(f"Data type: {t1w_data.dtype}")
    print(f"Intensity range: [{t1w_data.min():.2f}, {t1w_data.max():.2f}]")
    
    plot_3d_slices(t1w_data, title="Original T1w Image")
    plt.show()
else:
    print("Cannot proceed without input file.")

## Step 2: Run FreeSurfer Complete Pipeline

### Option A: Standard Pipeline (Ages 25+ months or Adults)

**Single command - processes everything!**

This runs the complete FreeSurfer pipeline including:
- Motion correction and intensity normalization
- Skull stripping and brain extraction
- Tissue segmentation (GM/WM/CSF)
- Subcortical structure segmentation
- White matter and pial surface reconstruction
- Cortical parcellation (Desikan-Killiany atlas)
- Statistical measurements

**Processing time:** 6-20 hours (depending on hardware)

In [ ]:
print("""OPTION A: Standard FreeSurfer Pipeline (Recommended for 25+ months)
="*70)

# Command to run
cmd = f"""
export SUBJECTS_DIR={SUBJECTS_DIR}

recon-all \
  -i {INPUT_T1W} \
  -subjid {SUBJECT_ID} \
  -all \
  -parallel \
  -openmp 4

# Optional flags:
# -parallel: Use parallel processing (faster)
# -openmp 4: Use 4 CPU cores (adjust based on your system)
"""

print("Command to run:")
print(cmd)

print("\n" + "="*70)
print("TO RUN: Copy the command above to your terminal")
print("ESTIMATED TIME: 6-12 hours")
print("="*70)

# Uncomment below to run directly from notebook (not recommended for long jobs)
# import subprocess
# result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
# print(result.stdout)

### Option B: With Infant FreeSurfer Enhancement (Ages 0-24 months)

For infants, you can optionally use infant FreeSurfer for better subcortical segmentation.

**Two-stage process:**
1. Standard FreeSurfer (for format conversion and preprocessing)
2. Infant FreeSurfer (for age-appropriate segmentation)

**Note:** This is simpler than the full pipeline because we skip iBEATv2 merging!

**Processing time:** 10-20 hours

In [ ]:
if AGE_MONTHS is not None and AGE_MONTHS < 25:
    print(f"""OPTION B: Infant FreeSurfer Enhancement (Age: {AGE_MONTHS} months)
="*70)
    
    infant_cmd = f"""
# STEP 1: Run standard FreeSurfer first
export SUBJECTS_DIR={SUBJECTS_DIR}
recon-all -i {INPUT_T1W} -subjid {SUBJECT_ID} -all

# STEP 2: Convert to infant FreeSurfer format
IFS_DIR={SUBJECTS_DIR}/infant_fs
mkdir -p $IFS_DIR/{SUBJECT_ID}
mri_convert -i {SUBJECTS_DIR}/{SUBJECT_ID}/mri/orig.mgz \
            -o $IFS_DIR/{SUBJECT_ID}/mprage.nii.gz

# STEP 3: Run infant FreeSurfer (if available)
export FREESURFER_HOME=/path/to/infant_freesurfer
source $FREESURFER_HOME/SetUpFreeSurfer.sh
export SUBJECTS_DIR=$IFS_DIR

infant_recon_all --s {SUBJECT_ID} --age {AGE_MONTHS}

# STEP 4: Use infant FreeSurfer segmentation with standard FreeSurfer surfaces
# Copy infant segmentation to standard FreeSurfer subject
cp $IFS_DIR/{SUBJECT_ID}/mri/aseg.mgz \
   {SUBJECTS_DIR}/{SUBJECT_ID}/mri/aseg.infant.mgz

cp $IFS_DIR/{SUBJECT_ID}/mri/brainmask.mgz \
   {SUBJECTS_DIR}/{SUBJECT_ID}/mri/brainmask.infant.mgz

# Use infant segmentation for surface reconstruction
export SUBJECTS_DIR={SUBJECTS_DIR}
recon-all -subjid {SUBJECT_ID} \
          -autorecon2 -autorecon3 \
          -aseg aseg.infant.mgz
"""
    
    print("Command to run:")
    print(infant_cmd)
    
    print("\n" + "="*70)
    print("TO RUN: Copy the command above to your terminal")
    print("ESTIMATED TIME: 10-20 hours")
    print("NOTE: Requires infant FreeSurfer to be installed")
    print("="*70)
else:
    print(f"Age is {AGE_MONTHS} months - standard FreeSurfer (Option A) is recommended")

## Step 3: Monitor Processing

FreeSurfer creates log files you can check to monitor progress.

In [ ]:
import os
from datetime import datetime

def check_freesurfer_status(subjects_dir, subject_id):
    """Check the status of FreeSurfer processing."""
    subject_dir = os.path.join(subjects_dir, subject_id)
    
    if not os.path.exists(subject_dir):
        print(f"❌ Subject directory not found: {subject_dir}")
        print("   FreeSurfer has not been run yet.")
        return
    
    print(f"✓ Subject directory exists: {subject_dir}\n")
    
    # Check log files
    log_file = os.path.join(subject_dir, "scripts", "recon-all.log")
    error_file = os.path.join(subject_dir, "scripts", "recon-all.error")
    
    if os.path.exists(error_file):
        print("⚠️  Error file exists - checking for errors...")
        with open(error_file, 'r') as f:
            error_content = f.read()
            if error_content.strip():
                print("❌ ERRORS FOUND:")
                print(error_content[-1000:])  # Last 1000 chars
            else:
                print("✓ No errors in error file")
    
    if os.path.exists(log_file):
        print(f"\n📄 Log file: {log_file}")
        
        # Get last 20 lines of log
        with open(log_file, 'r') as f:
            lines = f.readlines()
            print("\nLast 20 lines of log:")
            print("="*70)
            for line in lines[-20:]:
                print(line.rstrip())
            print("="*70)
    
    # Check for key output files
    print("\n📊 Checking output files:")
    
    key_files = {
        "Original image": "mri/orig.mgz",
        "Brain mask": "mri/brainmask.mgz",
        "Segmentation": "mri/aseg.mgz",
        "LH white surface": "surf/lh.white",
        "RH white surface": "surf/rh.white",
        "LH pial surface": "surf/lh.pial",
        "RH pial surface": "surf/rh.pial",
        "LH parcellation": "label/lh.aparc.annot",
        "RH parcellation": "label/rh.aparc.annot",
        "Segmentation stats": "stats/aseg.stats",
        "LH cortical stats": "stats/lh.aparc.stats",
        "RH cortical stats": "stats/rh.aparc.stats",
    }
    
    for name, rel_path in key_files.items():
        full_path = os.path.join(subject_dir, rel_path)
        if os.path.exists(full_path):
            size = os.path.getsize(full_path)
            mod_time = datetime.fromtimestamp(os.path.getmtime(full_path))
            print(f"  ✓ {name:25s} [{size:>10,} bytes] {mod_time.strftime('%Y-%m-%d %H:%M')}")
        else:
            print(f"  ❌ {name:25s} [NOT FOUND]")
    
    # Check if completed
    if all(os.path.exists(os.path.join(subject_dir, p)) for p in key_files.values()):
        print("\n" + "="*70)
        print("🎉 FreeSurfer processing appears COMPLETE!")
        print("="*70)
    else:
        print("\n" + "="*70)
        print("⏳ FreeSurfer processing still in progress...")
        print("="*70)

# Run the check
check_freesurfer_status(SUBJECTS_DIR, SUBJECT_ID)

## Step 4: Visualize Results

Once FreeSurfer completes, visualize the outputs.

In [ ]:
def plot_segmentation(seg_data, title="Segmentation", figsize=(15, 5)):
    """Plot segmentation with color map."""
    n_labels = int(seg_data.max()) + 1
    cmap = plt.cm.get_cmap('tab20', n_labels)
    
    mid_x = seg_data.shape[0] // 2
    mid_y = seg_data.shape[1] // 2
    mid_z = seg_data.shape[2] // 2
    
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    axes[0].imshow(np.rot90(seg_data[mid_x, :, :]), cmap=cmap, interpolation='nearest')
    axes[0].set_title('Sagittal')
    axes[0].axis('off')
    
    axes[1].imshow(np.rot90(seg_data[:, mid_y, :]), cmap=cmap, interpolation='nearest')
    axes[1].set_title('Coronal')
    axes[1].axis('off')
    
    axes[2].imshow(np.rot90(seg_data[:, :, mid_z]), cmap=cmap, interpolation='nearest')
    axes[2].set_title('Axial')
    axes[2].axis('off')
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig

# Load and display segmentation
aseg_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "mri", "aseg.mgz")

if os.path.exists(aseg_path):
    print("Loading FreeSurfer segmentation...")
    aseg_data, _, _ = load_nifti(aseg_path)
    
    print(f"Number of unique labels: {len(np.unique(aseg_data))}")
    print(f"Label range: {aseg_data.min():.0f} to {aseg_data.max():.0f}")
    
    plot_segmentation(aseg_data, title="FreeSurfer Tissue Segmentation")
    plt.show()
    
    # Volume statistics
    unique, counts = np.unique(aseg_data, return_counts=True)
    print("\nTop 10 structures by volume:")
    sorted_idx = np.argsort(counts)[::-1][:10]
    for idx in sorted_idx:
        label = unique[idx]
        count = counts[idx]
        print(f"  Label {label:3.0f}: {count:8d} voxels")
else:
    print(f"Segmentation not found at: {aseg_path}")
    print("FreeSurfer has not completed yet.")

### Visualize Brain Mask

In [ ]:
brainmask_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "mri", "brainmask.mgz")

if os.path.exists(brainmask_path):
    print("Loading brain mask...")
    mask_data, _, _ = load_nifti(brainmask_path)
    
    plot_3d_slices(mask_data, title="Brain Mask", cmap='hot')
    plt.show()
    
    brain_volume = np.sum(mask_data > 0)
    print(f"\nBrain volume: {brain_volume:,} voxels")
else:
    print(f"Brain mask not found at: {brainmask_path}")

### Visualize Cortical Surfaces

In [ ]:
def load_surface(filepath):
    """Load FreeSurfer surface file."""
    try:
        from nibabel.freesurfer import read_geometry
        vertices, faces = read_geometry(filepath)
        return vertices, faces
    except Exception as e:
        print(f"Error loading surface: {e}")
        return None, None

def plot_surface_3d(vertices, faces, title="Surface", color='lightblue', alpha=0.8):
    """Plot 3D surface mesh."""
    from mpl_toolkits.mplot3d.art3d import Poly3DCollection
    
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    mesh = Poly3DCollection(vertices[faces], alpha=alpha, facecolor=color, edgecolor='none')
    ax.add_collection3d(mesh)
    
    ax.set_xlim(vertices[:, 0].min(), vertices[:, 0].max())
    ax.set_ylim(vertices[:, 1].min(), vertices[:, 1].max())
    ax.set_zlim(vertices[:, 2].min(), vertices[:, 2].max())
    
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.view_init(elev=20, azim=45)
    
    return fig, ax

# Load surfaces
lh_pial_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "surf", "lh.pial")
rh_pial_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "surf", "rh.pial")

if os.path.exists(lh_pial_path) and os.path.exists(rh_pial_path):
    print("Loading pial surfaces...")
    lh_verts, lh_faces = load_surface(lh_pial_path)
    rh_verts, rh_faces = load_surface(rh_pial_path)
    
    if lh_verts is not None:
        print(f"LH surface: {len(lh_verts):,} vertices, {len(lh_faces):,} faces")
        print(f"RH surface: {len(rh_verts):,} vertices, {len(rh_faces):,} faces")
        
        plot_surface_3d(lh_verts, lh_faces, "Left Hemisphere Pial Surface", color='coral')
        plt.show()
        
        plot_surface_3d(rh_verts, rh_faces, "Right Hemisphere Pial Surface", color='steelblue')
        plt.show()
else:
    print("Surface files not found. FreeSurfer has not completed yet.")

## Step 5: Extract Measurements

In [ ]:
import pandas as pd

def parse_aseg_stats(stats_file):
    """Parse FreeSurfer aseg.stats file."""
    data = []
    with open(stats_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.strip().split()
            if len(parts) >= 5:
                try:
                    index = int(parts[0])
                    volume = float(parts[3])
                    name = parts[4] if len(parts) > 4 else "Unknown"
                    data.append({'Structure': name, 'Volume_mm3': volume})
                except ValueError:
                    continue
    return pd.DataFrame(data)

# Load subcortical volumes
aseg_stats_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "stats", "aseg.stats")

if os.path.exists(aseg_stats_path):
    print("📊 Subcortical Structure Volumes:\n")
    df = parse_aseg_stats(aseg_stats_path)
    
    # Display key structures
    key_structures = ['Left-Thalamus', 'Right-Thalamus', 
                      'Left-Caudate', 'Right-Caudate',
                      'Left-Putamen', 'Right-Putamen',
                      'Left-Hippocampus', 'Right-Hippocampus',
                      'Left-Cerebral-White-Matter', 'Right-Cerebral-White-Matter',
                      'Left-Cerebral-Cortex', 'Right-Cerebral-Cortex']
    
    for struct in key_structures:
        row = df[df['Structure'] == struct]
        if not row.empty:
            volume = row.iloc[0]['Volume_mm3']
            print(f"  {struct:35s}: {volume:10.2f} mm³")
    
    # Display full table
    print("\nComplete table:")
    display(df.head(20))
else:
    print(f"Stats file not found at: {aseg_stats_path}")

### Cortical Statistics

In [ ]:
def parse_aparc_stats(stats_file):
    """Parse FreeSurfer aparc.stats file."""
    data = []
    with open(stats_file, 'r') as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.strip().split()
            if len(parts) >= 5:
                try:
                    name = parts[0]
                    area = float(parts[2])
                    thickness = float(parts[4])
                    data.append({
                        'Region': name,
                        'Area_mm2': area,
                        'Thickness_mm': thickness
                    })
                except ValueError:
                    continue
    return pd.DataFrame(data)

# Load cortical measurements
lh_aparc_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "stats", "lh.aparc.stats")
rh_aparc_path = os.path.join(SUBJECTS_DIR, SUBJECT_ID, "stats", "rh.aparc.stats")

if os.path.exists(lh_aparc_path):
    print("📊 Left Hemisphere Cortical Measurements:\n")
    lh_df = parse_aparc_stats(lh_aparc_path)
    display(lh_df.head(10))
    
    print(f"\nAverage cortical thickness (LH): {lh_df['Thickness_mm'].mean():.3f} mm")
    print(f"Total cortical area (LH): {lh_df['Area_mm2'].sum():.2f} mm²")

if os.path.exists(rh_aparc_path):
    print("\n📊 Right Hemisphere Cortical Measurements:\n")
    rh_df = parse_aparc_stats(rh_aparc_path)
    display(rh_df.head(10))
    
    print(f"\nAverage cortical thickness (RH): {rh_df['Thickness_mm'].mean():.3f} mm")
    print(f"Total cortical area (RH): {rh_df['Area_mm2'].sum():.2f} mm²")

## Summary: Simplified vs Full Pipeline

### What You Get with Simplified Pipeline:
✅ Complete FreeSurfer processing
✅ Brain segmentation (FreeSurfer's native)
✅ Cortical surface reconstruction
✅ Cortical parcellation (34 regions/hemisphere)
✅ All morphometric measurements
✅ Optional infant FreeSurfer enhancement
✅ **No external dependencies** (except FreeSurfer)

### What You Miss (vs Full Pipeline):
❌ iBEATv2's superior cortical GM/WM boundary detection (infants only)
❌ Merged iBEATv2 + iFS segmentation
❌ Optimal accuracy for cortical segmentation in infants <25 months

### Accuracy Comparison:

| Age Group | Simplified Pipeline | Full Pipeline | Difference |
|-----------|---------------------|---------------|------------|
| **0-12 months** | Good | Excellent | Noticeable (~10-15% cortical accuracy) |
| **12-24 months** | Good | Very Good | Small (~5-10% cortical accuracy) |
| **25-50 months** | Excellent | Excellent | Negligible (<2%) |
| **50+ months** | Excellent | Excellent | None |

### Recommendation:
- **Ages 0-24 months + critical cortical analysis**: Use full pipeline (if iBEATv2 available)
- **Ages 0-24 months + subcortical focus**: Simplified pipeline is sufficient
- **Ages 25+ months**: Simplified pipeline is excellent
- **Just getting started**: Use simplified pipeline!

### Running Both Pipelines:
You can process the same subject with both pipelines and compare:
```bash
# Simplified: outputs to freesurfer_simplified/
# Full: outputs to freesurfer_output/
```

Then use visualization tools like FreeView to compare the segmentations:
```bash
freeview \
  -v freesurfer_simplified/sub-01/mri/aseg.mgz \
  -v freesurfer_output/sub-01/mri/aseg.presurf.mgz:colormap=lut:opacity=0.5
```